# SwiGLU Allocation Study

This notebook extends the optimized SwiGLU workflow with 25% and 50% model-impact probes, representative-block width responses, entropy-regularized allocation, controlled minimum-width ablation, and finalist recovery.

#### setup

In [ ]:
from copy import deepcopy
from dataclasses import asdict
from datetime import datetime, timezone
import json
import math
from pathlib import Path
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlp_replacement.capture import collect_modules_io
from mlp_replacement.compression.recovery import (
    cache_teacher_logits,
    mean_cache_loss,
    recover_replacements,
)
from mlp_replacement.compression.surgery import (
    temporary_replacement,
    temporary_replacements,
)
from mlp_replacement.config import (
    CaptureConfig,
    DataConfig,
    DatasetSpec,
    ModelConfig,
    OperatorConfig,
    RecoveryConfig,
)
from mlp_replacement.data import (
    DataLoaders,
    contiguous_token_windows,
    load_text_dataset,
    make_token_loader,
    sample_partitioned_windows,
)
from mlp_replacement.evaluation.footprint import parameter_footprint
from mlp_replacement.evaluation.language_model import (
    evaluate_language_model,
)
from mlp_replacement.evaluation.operator import evaluate_operator
from mlp_replacement.model import (
    discover_mlp_blocks,
    load_model_and_tokenizer,
)
from mlp_replacement.operators import (
    GatedMLPReplacement,
    fit_operator,
    initialize_gated_mlp_from_teacher,
)
from mlp_replacement.runlog import environment_record

In [ ]:
def report_memory(stage):
    status_path = Path('/proc/self/status')
    if status_path.exists():
        status = dict(
            line.split(':', 1)
            for line in status_path.read_text().splitlines()
        )
        rss = int(status['VmRSS'].split()[0]) / 1024**2
        peak = int(status['VmHWM'].split()[0]) / 1024**2
        print(f'{stage}: RAM {rss:.2f} GiB, peak {peak:.2f} GiB')


def synchronize_cuda(active_device):
    if (
        active_device is not None
        and torch.device(active_device).type == 'cuda'
    ):
        torch.cuda.synchronize(active_device)

In [ ]:
SWIGLU_2_EXECUTION_MODE = 'run'
RUN_SWIGLU_2 = SWIGLU_2_EXECUTION_MODE == 'run'
REFERENCE_ARTIFACT_SCHEMA_VERSION = 3
SWIGLU_2_ARTIFACT_SCHEMA_VERSION = 1
REFERENCE_ARTIFACT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'notebook-model-study'
    / 'swiglu-compression-optimized.json'
)
SWIGLU_2_ARTIFACT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'notebook-model-study'
    / 'swiglu-2.json'
)

PROBE_WIDTH_RATIOS = (0.25, 0.5)
REFERENCE_WIDTH_RATIO = 0.5
WIDTH_SWEEP_RATIOS = (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)
ALLOCATION_TEMPERATURES = (1.0, 2.0, 4.0)
BOUNDARY_MINIMUM_RETENTIONS = (0.3, 0.4)
ALLOCATION_SELECTION_BATCHES = 12
SCREENING_CALIBRATION_BATCHES = 96
ALLOCATION_SCORE_NAMES = (
    'canonical_bi',
    'residual_aware_mlp_bi',
    'singleton_kl_w25',
    'singleton_kl_w50',
    'singleton_loss_delta_w25',
    'singleton_loss_delta_w50',
)

In [ ]:
reference_artifact = json.loads(
    REFERENCE_ARTIFACT_PATH.read_text(encoding='utf-8')
)
if (
    reference_artifact.get('schema_version')
    != REFERENCE_ARTIFACT_SCHEMA_VERSION
):
    raise ValueError('Incompatible optimized SwiGLU artifact')

swiglu_2_artifact = (
    None
    if RUN_SWIGLU_2
    else json.loads(
        SWIGLU_2_ARTIFACT_PATH.read_text(encoding='utf-8')
    )
)
if (
    not RUN_SWIGLU_2
    and swiglu_2_artifact.get('schema_version')
    != SWIGLU_2_ARTIFACT_SCHEMA_VERSION
):
    raise ValueError('Incompatible SwiGLU allocation artifact')

In [ ]:
reference_config = reference_artifact['configuration']
model_values = reference_config['model']
resolved_revision = (
    model_values.get('resolved_revision')
    or model_values['revision']
)
model_config = ModelConfig(
    model_id=model_values['model_id'],
    revision=resolved_revision,
    tokenizer_revision=(
        model_values.get('tokenizer_revision')
        or resolved_revision
    ),
    device=model_values['device'],
    dtype=model_values['dtype'],
    trust_remote_code=model_values['trust_remote_code'],
)

data_values = dict(reference_config['data'])
for source_name in (
    'calibration_source',
    'model_validation_source',
    'test_source',
):
    data_values[source_name] = DatasetSpec(
        **data_values[source_name]
    )
data_config = DataConfig(**data_values)

capture_values = dict(reference_config['capture'])
CAPTURE_GROUP_SIZE = int(capture_values.pop('module_group_size'))
capture_config = CaptureConfig(**capture_values)
training_config = OperatorConfig(
    **reference_config['operator_training']['complete_method']
)
recovery_values = dict(reference_config['recovery'])
recovery_values.pop('trainable_scope', None)
recovery_config = RecoveryConfig(**recovery_values)

SEED = int(data_config.seed)
TARGET_MLP_SPARSITY = float(
    reference_config['target_mlp_sparsity']
)
ELIGIBLE_LAYERS = tuple(reference_config['eligible_layers'])
PROTECTED_LAYERS = tuple(reference_config['protected_layers'])

pairs_per_batch = data_config.batch_size * data_config.sequence_length
historical_calibration_pairs = int(
    reference_config['methodologies']['historical_random'][
        'calibration_pairs'
    ]
)
if historical_calibration_pairs % pairs_per_batch:
    raise ValueError('Historical calibration pairs do not form full batches')
BASE_CALIBRATION_BATCHES = historical_calibration_pairs // pairs_per_batch
ADDITIONAL_CALIBRATION_BATCHES = (
    data_config.num_calibration_batches - BASE_CALIBRATION_BATCHES
)
if ADDITIONAL_CALIBRATION_BATCHES < 0:
    raise ValueError('Optimized calibration budget is smaller than the baseline')
PARTITION_BATCHES = {
    'calibration': BASE_CALIBRATION_BATCHES,
    'operator_validation': data_config.num_operator_validation_batches,
    'recovery': data_config.num_recovery_batches,
    'recovery_validation': (
        data_config.num_recovery_validation_batches
    ),
    'additional_calibration': ADDITIONAL_CALIBRATION_BATCHES,
    'allocation_selection': ALLOCATION_SELECTION_BATCHES,
}

In [ ]:
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')
RUN_STARTED = perf_counter() if RUN_SWIGLU_2 else None
runtime_rows = (
    []
    if RUN_SWIGLU_2
    else list(swiglu_2_artifact['results']['runtime'])
)

#### model and data

In [ ]:
if RUN_SWIGLU_2:
    model, tokenizer = load_model_and_tokenizer(model_config)
    device = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype
    mlp_blocks = discover_mlp_blocks(model)
    mlp_blocks_by_layer = {
        block.index: block for block in mlp_blocks
    }
    if not set(ELIGIBLE_LAYERS).issubset(mlp_blocks_by_layer):
        raise ValueError(
            'Reference eligible layers do not match the model'
        )
else:
    model = tokenizer = device = model_dtype = None
    mlp_blocks = mlp_blocks_by_layer = None

In [ ]:
if RUN_SWIGLU_2:
    calibration_data = load_text_dataset(
        data_config.calibration_source
    )
    partitioned_windows = sample_partitioned_windows(
        calibration_data,
        tokenizer,
        {
            name: batches * data_config.batch_size
            for name, batches in PARTITION_BATCHES.items()
        },
        data_config.sequence_length,
        data_config.seed,
        data_config.calibration_source.text_column,
    )
    calibration_sequences = (
        partitioned_windows['calibration']
        + partitioned_windows['additional_calibration']
    )
    model_validation_data = load_text_dataset(
        data_config.model_validation_source
    )
    model_validation_sequences = contiguous_token_windows(
        model_validation_data,
        tokenizer,
        data_config.num_model_validation_batches
        * data_config.batch_size,
        data_config.sequence_length,
        data_config.model_validation_source.text_column,
    )
    allocation_selection_loader = make_token_loader(
        partitioned_windows['allocation_selection'],
        data_config.batch_size,
    )
    screening_calibration_loader = make_token_loader(
        calibration_sequences[
            :SCREENING_CALIBRATION_BATCHES
            * data_config.batch_size
        ],
        data_config.batch_size,
    )
    loaders = DataLoaders(
        calibration=make_token_loader(
            calibration_sequences, data_config.batch_size
        ),
        operator_validation=make_token_loader(
            partitioned_windows['operator_validation'],
            data_config.batch_size,
        ),
        recovery=make_token_loader(
            partitioned_windows['recovery'],
            data_config.batch_size,
        ),
        recovery_validation=make_token_loader(
            partitioned_windows['recovery_validation'],
            data_config.batch_size,
        ),
        model_validation=make_token_loader(
            model_validation_sequences, data_config.batch_size
        ),
        test=None,
    )
    del calibration_data, model_validation_data
else:
    partitioned_windows = loaders = None
    allocation_selection_loader = None
    screening_calibration_loader = None

In [ ]:
dense_reference_df = pd.DataFrame(
    reference_artifact['results']['dense_reference']
)
if RUN_SWIGLU_2:
    dense_footprint = parameter_footprint(model)
    expected_dense_parameters = int(
        dense_reference_df.iloc[0]['parameters']
    )
    if dense_footprint.parameters != expected_dense_parameters:
        raise ValueError('Loaded model footprint differs from the reference')
    original_mlp_parameters = {
        layer: sum(
            parameter.numel()
            for parameter in mlp_blocks_by_layer[
                layer
            ].module.parameters()
        )
        for layer in ELIGIBLE_LAYERS
    }
    eligible_mlp_parameters = sum(
        original_mlp_parameters.values()
    )
    fixed_model_parameters = (
        dense_footprint.parameters - eligible_mlp_parameters
    )
else:
    dense_footprint = original_mlp_parameters = None
    eligible_mlp_parameters = fixed_model_parameters = None

layer_groups = [
    ELIGIBLE_LAYERS[start:start + CAPTURE_GROUP_SIZE]
    for start in range(
        0, len(ELIGIBLE_LAYERS), CAPTURE_GROUP_SIZE
    )
]

#### Importance scoring

Fit 25% and 50% retained-width probes for every block, then measure their local fit and singleton model impact. Perplexity is reported, while the equivalent loss-delta ranking is used for allocation.

In [ ]:
if RUN_SWIGLU_2:
    reference_operators = {}
    reference_fitting_rows = []
    reference_history_rows = []
    aggressive_fitting_rows = []
    aggressive_history_rows = []
    aggressive_impact_rows = []
    teacher_neuron_rankings = {
        int(layer): torch.tensor(values, dtype=torch.long)
        for layer, values in reference_artifact['results'][
            'teacher_neuron_rankings'
        ].items()
    }
    allocation_selection_teacher_cache = cache_teacher_logits(
        model,
        allocation_selection_loader,
        ALLOCATION_SELECTION_BATCHES,
        device,
        recovery_config.cache_dtype,
    )
    allocation_selection_dense_metrics = evaluate_language_model(
        model,
        allocation_selection_loader,
        device,
        ALLOCATION_SELECTION_BATCHES,
    )
    probe_phase_started = perf_counter()
else:
    reference_operators = None
    teacher_neuron_rankings = None
    allocation_selection_teacher_cache = None
    allocation_selection_dense_metrics = None

In [ ]:
if RUN_SWIGLU_2:
    for group_index, layer_group in enumerate(layer_groups, start=1):
        print(
            f'Probe capture {group_index}/{len(layer_groups)} '
            f'| {layer_group}'
        )
        group_paths = [
            mlp_blocks_by_layer[layer].path
            for layer in layer_group
        ]
        capture_started = perf_counter()
        training_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            loaders.calibration,
            data_config.num_calibration_batches,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        validation_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            loaders.operator_validation,
            data_config.num_operator_validation_batches,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        runtime_rows.append({
            'stage': 'probe_capture',
            'group': group_index,
            'seconds': perf_counter() - capture_started,
        })
        report_memory(f'After probe capture {group_index}')

        for layer in layer_group:
            print(f'Fitting width-0.5 reference | layer {layer}')
            block = mlp_blocks_by_layer[layer]
            training_pairs = training_pairs_by_path[block.path]
            validation_pairs = validation_pairs_by_path[block.path]
            original_width = block.module.up_proj.out_features
            replacement_width = round(
                original_width * REFERENCE_WIDTH_RATIO
            )
            selected_indices = teacher_neuron_rankings[layer][
                :replacement_width
            ].sort().values
            module = GatedMLPReplacement(
                training_pairs.hidden_size, replacement_width
            ).to(device)
            initialize_gated_mlp_from_teacher(
                module, block.module, selected_indices
            )
            initial_metrics = evaluate_operator(
                module,
                validation_pairs,
                device,
                training_config.batch_size,
            )
            synchronize_cuda(device)
            fit_started = perf_counter()
            fit = fit_operator(
                module,
                training_pairs,
                validation_pairs,
                training_config,
                device,
            )
            synchronize_cuda(device)
            fit_seconds = perf_counter() - fit_started
            metrics = evaluate_operator(
                fit.module,
                validation_pairs,
                device,
                training_config.batch_size,
            )
            reference_operators[layer] = fit.module.to(
                device='cpu', dtype=model_dtype
            )
            reference_fitting_rows.append({
                'policy': 'uniform',
                'layer': layer,
                'probe_width_ratio': REFERENCE_WIDTH_RATIO,
                'replacement_width': replacement_width,
                'replacement_width_ratio': (
                    replacement_width / original_width
                ),
                'replacement_parameters': sum(
                    parameter.numel()
                    for parameter in fit.module.parameters()
                ),
                'initial_local_mse': initial_metrics.mse,
                'initial_local_relative_mse': (
                    initial_metrics.relative_mse
                ),
                'initial_local_cosine': (
                    initial_metrics.cosine_similarity
                ),
                'best_epoch': fit.best_epoch,
                'epochs_completed': len(fit.history),
                'local_mse': metrics.mse,
                'local_relative_mse': metrics.relative_mse,
                'local_cosine': metrics.cosine_similarity,
                'fit_seconds': fit_seconds,
            })
            for epoch in fit.history:
                reference_history_rows.append({
                    'policy': 'uniform',
                    'layer': layer,
                    'probe_width_ratio': REFERENCE_WIDTH_RATIO,
                    'epoch': epoch.epoch,
                    'train_mse': epoch.train_mse,
                    'validation_mse': epoch.validation_mse,
                    'learning_rate': epoch.learning_rate,
                })

            aggressive_width_ratio = PROBE_WIDTH_RATIOS[0]
            aggressive_width = max(
                1, round(original_width * aggressive_width_ratio)
            )
            print(
                f'Fitting width-0.25 probe | layer {layer}'
            )
            selected_indices = teacher_neuron_rankings[layer][
                :aggressive_width
            ].sort().values
            aggressive_module = GatedMLPReplacement(
                training_pairs.hidden_size, aggressive_width
            ).to(device)
            initialize_gated_mlp_from_teacher(
                aggressive_module, block.module, selected_indices
            )
            aggressive_initial_metrics = evaluate_operator(
                aggressive_module,
                validation_pairs,
                device,
                training_config.batch_size,
            )
            synchronize_cuda(device)
            aggressive_fit_started = perf_counter()
            aggressive_fit = fit_operator(
                aggressive_module,
                training_pairs,
                validation_pairs,
                training_config,
                device,
            )
            synchronize_cuda(device)
            aggressive_fit_seconds = (
                perf_counter() - aggressive_fit_started
            )
            aggressive_metrics = evaluate_operator(
                aggressive_fit.module,
                validation_pairs,
                device,
                training_config.batch_size,
            )
            synchronize_cuda(device)
            aggressive_evaluation_started = perf_counter()
            with temporary_replacement(
                model, layer, aggressive_fit.module
            ):
                aggressive_model_metrics = evaluate_language_model(
                    model,
                    allocation_selection_loader,
                    device,
                    ALLOCATION_SELECTION_BATCHES,
                )
                aggressive_kl = mean_cache_loss(
                    model,
                    allocation_selection_teacher_cache,
                    recovery_config.temperature,
                    device,
                )
            synchronize_cuda(device)
            aggressive_fitting_rows.append({
                'layer': layer,
                'probe_width_ratio': aggressive_width_ratio,
                'replacement_width': aggressive_width,
                'replacement_width_ratio': (
                    aggressive_width / original_width
                ),
                'replacement_parameters': sum(
                    parameter.numel()
                    for parameter in aggressive_fit.module.parameters()
                ),
                'initial_local_mse': (
                    aggressive_initial_metrics.mse
                ),
                'initial_local_relative_mse': (
                    aggressive_initial_metrics.relative_mse
                ),
                'initial_local_cosine': (
                    aggressive_initial_metrics.cosine_similarity
                ),
                'best_epoch': aggressive_fit.best_epoch,
                'epochs_completed': len(aggressive_fit.history),
                'local_mse': aggressive_metrics.mse,
                'local_relative_mse': (
                    aggressive_metrics.relative_mse
                ),
                'local_cosine': (
                    aggressive_metrics.cosine_similarity
                ),
                'fit_seconds': aggressive_fit_seconds,
            })
            aggressive_impact_rows.append({
                'layer': layer,
                'probe_width_ratio': aggressive_width_ratio,
                'singleton_kl': aggressive_kl,
                'singleton_loss': aggressive_model_metrics.loss,
                'singleton_loss_delta': (
                    aggressive_model_metrics.loss
                    - allocation_selection_dense_metrics.loss
                ),
                'singleton_perplexity': (
                    aggressive_model_metrics.perplexity
                ),
                'singleton_perplexity_delta': (
                    aggressive_model_metrics.perplexity
                    - allocation_selection_dense_metrics.perplexity
                ),
                'evaluation_seconds': (
                    perf_counter()
                    - aggressive_evaluation_started
                ),
            })
            for epoch in aggressive_fit.history:
                aggressive_history_rows.append({
                    'layer': layer,
                    'probe_width_ratio': aggressive_width_ratio,
                    'epoch': epoch.epoch,
                    'train_mse': epoch.train_mse,
                    'validation_mse': epoch.validation_mse,
                    'learning_rate': epoch.learning_rate,
                })
            aggressive_fit.module.to(
                device='cpu', dtype=model_dtype
            )

        module = fit = metrics = initial_metrics = None
        aggressive_module = aggressive_fit = None
        aggressive_metrics = aggressive_initial_metrics = None
        training_pairs = validation_pairs = None
        training_pairs_by_path.clear()
        validation_pairs_by_path.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        report_memory(f'After probe group {group_index}')

    reference_fitting_df = pd.DataFrame(reference_fitting_rows)
    reference_history_df = pd.DataFrame(reference_history_rows)
    aggressive_fitting_df = pd.DataFrame(
        aggressive_fitting_rows
    )
    aggressive_history_df = pd.DataFrame(
        aggressive_history_rows
    )
    runtime_rows.append({
        'stage': 'dual_width_probes_total',
        'group': None,
        'seconds': perf_counter() - probe_phase_started,
    })
else:
    probe_fitting_df = pd.DataFrame(
        swiglu_2_artifact['results']['probe_operator_fitting']
    )
    probe_history_df = pd.DataFrame(
        swiglu_2_artifact['results'][
            'probe_operator_training_history'
        ]
    )
    probe_impact_df = pd.DataFrame(
        swiglu_2_artifact['results']['probe_model_impact']
    )
    reference_fitting_df = probe_fitting_df.query(
        'probe_width_ratio == 0.5'
    ).reset_index(drop=True)
    reference_history_df = probe_history_df.query(
        'probe_width_ratio == 0.5'
    ).reset_index(drop=True)
    reference_impact_df = probe_impact_df.query(
        'probe_width_ratio == 0.5'
    ).reset_index(drop=True)
    aggressive_fitting_df = probe_fitting_df.query(
        'probe_width_ratio == 0.25'
    ).reset_index(drop=True)
    aggressive_history_df = probe_history_df.query(
        'probe_width_ratio == 0.25'
    ).reset_index(drop=True)
    aggressive_impact_df = probe_impact_df.query(
        'probe_width_ratio == 0.25'
    ).reset_index(drop=True)

In [ ]:
if RUN_SWIGLU_2:
    score_phase_started = perf_counter()
    singleton_score_rows = []

    for layer in ELIGIBLE_LAYERS:
        print(f'Singleton model-impact scoring | layer {layer}')
        replacement = reference_operators[layer]
        synchronize_cuda(device)
        score_started = perf_counter()
        with temporary_replacement(model, layer, replacement):
            singleton_metrics = evaluate_language_model(
                model,
                allocation_selection_loader,
                device,
                ALLOCATION_SELECTION_BATCHES,
            )
            singleton_kl = mean_cache_loss(
                model,
                allocation_selection_teacher_cache,
                recovery_config.temperature,
                device,
            )
        synchronize_cuda(device)
        replacement.to(device='cpu', dtype=model_dtype)
        singleton_score_rows.append({
            'layer': layer,
            'probe_width_ratio': REFERENCE_WIDTH_RATIO,
            'singleton_kl': singleton_kl,
            'singleton_loss': singleton_metrics.loss,
            'singleton_loss_delta': (
                singleton_metrics.loss
                - allocation_selection_dense_metrics.loss
            ),
            'singleton_perplexity': singleton_metrics.perplexity,
            'singleton_perplexity_delta': (
                singleton_metrics.perplexity
                - allocation_selection_dense_metrics.perplexity
            ),
            'evaluation_seconds': perf_counter() - score_started,
        })

    reference_importance_df = pd.DataFrame(
        reference_artifact['results']['importance']
    )
    reference_impact_df = pd.DataFrame(singleton_score_rows)
    aggressive_impact_df = pd.DataFrame(aggressive_impact_rows)
    probe_fitting_df = pd.concat(
        [aggressive_fitting_df, reference_fitting_df],
        ignore_index=True,
    )
    probe_history_df = pd.concat(
        [aggressive_history_df, reference_history_df],
        ignore_index=True,
    )
    probe_impact_df = pd.concat(
        [aggressive_impact_df, reference_impact_df],
        ignore_index=True,
    )
    candidate_score_df = reference_importance_df.copy()
    for probe_width_ratio, suffix in (
        (0.25, 'w25'),
        (0.5, 'w50'),
    ):
        local_rows = (
            probe_fitting_df[
                probe_fitting_df['probe_width_ratio']
                == probe_width_ratio
            ][[
                'layer',
                'initial_local_relative_mse',
                'local_relative_mse',
                'local_cosine',
            ]]
            .rename(columns={
                'initial_local_relative_mse': (
                    f'initial_local_relative_mse_{suffix}'
                ),
                'local_relative_mse': (
                    f'local_relative_mse_{suffix}'
                ),
                'local_cosine': f'local_cosine_{suffix}',
            })
        )
        impact_rows = (
            probe_impact_df[
                probe_impact_df['probe_width_ratio']
                == probe_width_ratio
            ][[
                'layer',
                'singleton_kl',
                'singleton_loss',
                'singleton_loss_delta',
                'singleton_perplexity',
                'singleton_perplexity_delta',
            ]]
            .rename(columns={
                column: f'{column}_{suffix}'
                for column in (
                    'singleton_kl',
                    'singleton_loss',
                    'singleton_loss_delta',
                    'singleton_perplexity',
                    'singleton_perplexity_delta',
                )
            })
        )
        candidate_score_df = (
            candidate_score_df
            .merge(local_rows, on='layer', validate='one_to_one')
            .merge(impact_rows, on='layer', validate='one_to_one')
        )
    for score_name in ALLOCATION_SCORE_NAMES:
        candidate_score_df[f'{score_name}_rank'] = (
            candidate_score_df[score_name]
            .rank(method='min', ascending=False)
            .astype(int)
        )
    runtime_rows.append({
        'stage': 'candidate_scores_total',
        'group': None,
        'seconds': perf_counter() - score_phase_started,
    })
else:
    allocation_selection_teacher_cache = None
    allocation_selection_dense_metrics = None
    candidate_score_df = pd.DataFrame(
        swiglu_2_artifact['results']['candidate_scores']
    )

reporting

In [ ]:
probe_summary_df = (
    probe_fitting_df.groupby(
        'probe_width_ratio', as_index=False
    )
    .agg(
        blocks=('layer', 'size'),
        mean_nmse=('local_relative_mse', 'mean'),
        worst_nmse=('local_relative_mse', 'max'),
        mean_epochs=('epochs_completed', 'mean'),
        total_fit_minutes=(
            'fit_seconds', lambda values: values.sum() / 60
        ),
    )
)
display(probe_summary_df)

score_report_df = candidate_score_df[[
    'layer',
    'canonical_bi',
    'residual_aware_mlp_bi',
    'singleton_kl_w25',
    'singleton_kl_w50',
    'singleton_loss_delta_w25',
    'singleton_loss_delta_w50',
    'singleton_perplexity_w25',
    'singleton_perplexity_w50',
]].copy()
display(score_report_df)
display(
    candidate_score_df[list(ALLOCATION_SCORE_NAMES)]
    .corr(method='spearman')
)

score_plot_df = candidate_score_df.melt(
    id_vars='layer',
    value_vars=list(ALLOCATION_SCORE_NAMES),
    var_name='score',
    value_name='value',
)
score_plot = sns.relplot(
    data=score_plot_df,
    x='layer',
    y='value',
    col='score',
    col_wrap=2,
    kind='line',
    marker='o',
    facet_kws={'sharey': False},
    height=3.5,
    aspect=1.35,
)
score_plot.set_axis_labels('Transformer block', 'Score')
plt.show()

probe_comparison_df = probe_impact_df.melt(
    id_vars=['layer', 'probe_width_ratio'],
    value_vars=['singleton_kl', 'singleton_loss_delta'],
    var_name='metric',
    value_name='value',
)
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for axis, metric, label in zip(
    axes,
    ('singleton_kl', 'singleton_loss_delta'),
    ('Teacher KL', 'Loss delta'),
):
    sns.lineplot(
        data=probe_comparison_df.query('metric == @metric'),
        x='probe_width_ratio',
        y='value',
        units='layer',
        estimator=None,
        marker='o',
        alpha=0.55,
        ax=axis,
    )
    axis.set(xlabel='Retained width', ylabel=label)
figure.suptitle('Block sensitivity at 25% and 50% width')
figure.tight_layout()
plt.show()

rank_shift_df = candidate_score_df[[
    'layer',
    'singleton_kl_w25_rank',
    'singleton_kl_w50_rank',
    'singleton_loss_delta_w25_rank',
    'singleton_loss_delta_w50_rank',
]].copy()
rank_shift_df['kl_rank_shift'] = (
    rank_shift_df['singleton_kl_w25_rank']
    - rank_shift_df['singleton_kl_w50_rank']
).abs()
rank_shift_df['loss_rank_shift'] = (
    rank_shift_df['singleton_loss_delta_w25_rank']
    - rank_shift_df['singleton_loss_delta_w50_rank']
).abs()
display(
    rank_shift_df.sort_values(
        ['kl_rank_shift', 'loss_rank_shift'], ascending=False
    ).reset_index(drop=True)
)

#### Block examination

In [ ]:
if RUN_SWIGLU_2:
    representative_blocks = {}
    representative_membership_rows = []
    for selection_basis, score_name in (
        ('kl50', 'singleton_kl_w50'),
        ('kl25', 'singleton_kl_w25'),
    ):
        score_order = candidate_score_df.sort_values(
            score_name
        ).reset_index(drop=True)
        cohort = {
            'best': int(score_order.iloc[0]['layer']),
            'middle': int(
                score_order.iloc[len(score_order) // 2]['layer']
            ),
            'worst': int(score_order.iloc[-1]['layer']),
        }
        representative_blocks[selection_basis] = cohort
        representative_membership_rows.extend(
            {
                'selection_basis': selection_basis,
                'impact_group': impact_group,
                'layer': layer,
                'selection_score': float(
                    candidate_score_df.loc[
                        candidate_score_df['layer'] == layer,
                        score_name,
                    ].iloc[0]
                ),
            }
            for impact_group, layer in cohort.items()
        )
    representative_membership_df = pd.DataFrame(
        representative_membership_rows
    )
    width_study_rows = []
    width_study_history_rows = []
    width_study_layers = tuple(dict.fromkeys(
        representative_membership_df['layer'].tolist()
    ))
    width_study_paths = [
        mlp_blocks_by_layer[layer].path
        for layer in width_study_layers
    ]
    width_capture_started = perf_counter()
    width_training_pairs = collect_modules_io(
        model,
        width_study_paths,
        loaders.calibration,
        data_config.num_calibration_batches,
        device,
        storage_device=capture_config.storage_device,
        storage_dtype=model_dtype,
    )
    width_validation_pairs = collect_modules_io(
        model,
        width_study_paths,
        loaders.operator_validation,
        data_config.num_operator_validation_batches,
        device,
        storage_device=capture_config.storage_device,
        storage_dtype=model_dtype,
    )
    runtime_rows.append({
        'stage': 'width_study_capture',
        'group': None,
        'seconds': perf_counter() - width_capture_started,
    })
    report_memory('After representative-block capture')

    for layer in width_study_layers:
        block = mlp_blocks_by_layer[layer]
        training_pairs = width_training_pairs[block.path]
        validation_pairs = width_validation_pairs[block.path]
        original_width = block.module.up_proj.out_features
        for width_ratio in WIDTH_SWEEP_RATIOS:
            replacement_width = max(
                1, round(original_width * width_ratio)
            )
            print(
                f'Width study | layer {layer} | '
                f'width {width_ratio:.1f}'
            )
            if math.isclose(width_ratio, REFERENCE_WIDTH_RATIO):
                probe_fit_row = reference_fitting_df[
                    reference_fitting_df['layer'] == layer
                ].iloc[0]
                probe_impact_row = reference_impact_df[
                    reference_impact_df['layer'] == layer
                ].iloc[0]
                width_study_rows.append({
                    'layer': layer,
                    'width_ratio': width_ratio,
                    'replacement_width': int(
                        probe_fit_row['replacement_width']
                    ),
                    'replacement_parameters': int(
                        probe_fit_row['replacement_parameters']
                    ),
                    'initial_local_relative_mse': (
                        probe_fit_row[
                            'initial_local_relative_mse'
                        ]
                    ),
                    'local_relative_mse': probe_fit_row[
                        'local_relative_mse'
                    ],
                    'local_cosine': probe_fit_row[
                        'local_cosine'
                    ],
                    'singleton_kl': probe_impact_row[
                        'singleton_kl'
                    ],
                    'singleton_loss': probe_impact_row[
                        'singleton_loss'
                    ],
                    'singleton_loss_delta': probe_impact_row[
                        'singleton_loss_delta'
                    ],
                    'singleton_perplexity': probe_impact_row[
                        'singleton_perplexity'
                    ],
                    'best_epoch': int(
                        probe_fit_row['best_epoch']
                    ),
                    'epochs_completed': int(
                        probe_fit_row['epochs_completed']
                    ),
                    'fit_seconds': probe_fit_row['fit_seconds'],
                    'evaluation_seconds': probe_impact_row[
                        'evaluation_seconds'
                    ],
                    'reused_probe': True,
                })
                reused_history = reference_history_df[
                    reference_history_df['layer'] == layer
                ]
                for epoch in reused_history.itertuples(index=False):
                    width_study_history_rows.append({
                        'layer': layer,
                        'width_ratio': width_ratio,
                        'epoch': epoch.epoch,
                        'train_mse': epoch.train_mse,
                        'validation_mse': epoch.validation_mse,
                        'learning_rate': epoch.learning_rate,
                    })
                continue
            selected_indices = teacher_neuron_rankings[layer][
                :replacement_width
            ].sort().values
            module = GatedMLPReplacement(
                training_pairs.hidden_size, replacement_width
            ).to(device)
            initialize_gated_mlp_from_teacher(
                module, block.module, selected_indices
            )
            initial_metrics = evaluate_operator(
                module,
                validation_pairs,
                device,
                training_config.batch_size,
            )
            synchronize_cuda(device)
            fit_started = perf_counter()
            fit = fit_operator(
                module,
                training_pairs,
                validation_pairs,
                training_config,
                device,
            )
            synchronize_cuda(device)
            fit_seconds = perf_counter() - fit_started
            local_metrics = evaluate_operator(
                fit.module,
                validation_pairs,
                device,
                training_config.batch_size,
            )
            synchronize_cuda(device)
            evaluation_started = perf_counter()
            with temporary_replacement(model, layer, fit.module):
                model_metrics = evaluate_language_model(
                    model,
                    allocation_selection_loader,
                    device,
                    ALLOCATION_SELECTION_BATCHES,
                )
                model_kl = mean_cache_loss(
                    model,
                    allocation_selection_teacher_cache,
                    recovery_config.temperature,
                    device,
                )
            synchronize_cuda(device)
            evaluation_seconds = perf_counter() - evaluation_started
            width_study_rows.append({
                'layer': layer,
                'width_ratio': width_ratio,
                'replacement_width': replacement_width,
                'replacement_parameters': sum(
                    parameter.numel()
                    for parameter in fit.module.parameters()
                ),
                'initial_local_relative_mse': (
                    initial_metrics.relative_mse
                ),
                'local_relative_mse': local_metrics.relative_mse,
                'local_cosine': local_metrics.cosine_similarity,
                'singleton_kl': model_kl,
                'singleton_loss': model_metrics.loss,
                'singleton_loss_delta': (
                    model_metrics.loss
                    - allocation_selection_dense_metrics.loss
                ),
                'singleton_perplexity': model_metrics.perplexity,
                'best_epoch': fit.best_epoch,
                'epochs_completed': len(fit.history),
                'fit_seconds': fit_seconds,
                'evaluation_seconds': evaluation_seconds,
                'reused_probe': False,
            })
            for epoch in fit.history:
                width_study_history_rows.append({
                    'layer': layer,
                    'width_ratio': width_ratio,
                    'epoch': epoch.epoch,
                    'train_mse': epoch.train_mse,
                    'validation_mse': epoch.validation_mse,
                    'learning_rate': epoch.learning_rate,
                })
            fit.module.to(device='cpu', dtype=model_dtype)
            module = fit = local_metrics = initial_metrics = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    width_study_df = pd.DataFrame(width_study_rows)
    width_study_history_df = pd.DataFrame(
        width_study_history_rows
    )
    width_training_pairs.clear()
    width_validation_pairs.clear()
    training_pairs = validation_pairs = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    report_memory('After representative-block study')
else:
    representative_blocks = {
        basis: {
            group: int(layer)
            for group, layer in cohort.items()
        }
        for basis, cohort in swiglu_2_artifact[
            'configuration'
        ]['representative_blocks'].items()
    }
    representative_membership_df = pd.DataFrame(
        swiglu_2_artifact['results'][
            'representative_block_membership'
        ]
    )
    width_study_df = pd.DataFrame(
        swiglu_2_artifact['results']['width_study']
    )
    width_study_history_df = pd.DataFrame(
        swiglu_2_artifact['results']['width_study_history']
    )

reporting

In [ ]:
representative_block_df = (
    representative_membership_df.sort_values(
        ['selection_basis', 'selection_score']
    ).reset_index(drop=True)
)
display(representative_block_df)

width_reporting_df = representative_membership_df.merge(
    width_study_df, on='layer', validate='many_to_many'
)

figure, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for column, selection_basis in enumerate(('kl50', 'kl25')):
    cohort_rows = width_reporting_df[
        width_reporting_df['selection_basis'] == selection_basis
    ]
    sns.lineplot(
        data=cohort_rows,
        x='width_ratio',
        y='local_relative_mse',
        hue='impact_group',
        marker='o',
        ax=axes[0, column],
    )
    sns.lineplot(
        data=cohort_rows,
        x='width_ratio',
        y='singleton_kl',
        hue='impact_group',
        marker='o',
        ax=axes[1, column],
    )
    axes[0, column].set_title(
        f'{selection_basis.upper()}-selected blocks'
    )
    axes[0, column].set_ylabel('Validation NMSE')
    axes[1, column].set_ylabel('Teacher KL')
    axes[1, column].set_xlabel('Retained SwiGLU width')
figure.suptitle('Width response of representative blocks')
figure.tight_layout()
plt.show()

#### Entropy regularization

In [ ]:
def rank_normalize_scores(scores):
    values = pd.Series(scores, dtype=float)
    if len(values) == 1:
        return {values.index[0]: 0.0}
    return (
        (values.rank(method='average') - 1) / (len(values) - 1)
    ).to_dict()


def allocate_swiglu_widths(
    policy,
    score_name=None,
    temperature=None,
    minimum_retention=None,
):
    scores = (
        None
        if score_name is None
        else dict(zip(
            candidate_score_df['layer'],
            candidate_score_df[score_name],
        ))
    )
    normalized = (
        None if scores is None else rank_normalize_scores(scores)
    )
    propensities = {
        layer: (
            1.0
            if normalized is None
            else math.exp(-normalized[layer] / temperature)
        )
        for layer in ELIGIBLE_LAYERS
    }
    target_removed = round(
        eligible_mlp_parameters * TARGET_MLP_SPARSITY
    )
    removal_limits = {
        layer: (
            original_mlp_parameters[layer]
            if minimum_retention is None
            else original_mlp_parameters[layer]
            * (1 - minimum_retention)
        )
        for layer in ELIGIBLE_LAYERS
    }
    allocated_removal = {layer: 0.0 for layer in ELIGIBLE_LAYERS}
    remaining_removal = float(target_removed)
    active_layers = set(ELIGIBLE_LAYERS)
    while active_layers:
        denominator = sum(
            original_mlp_parameters[layer] * propensities[layer]
            for layer in active_layers
        )
        proposed = {
            layer: remaining_removal
            * original_mlp_parameters[layer]
            * propensities[layer]
            / denominator
            for layer in active_layers
        }
        capped_layers = [
            layer
            for layer in active_layers
            if proposed[layer] > removal_limits[layer]
        ]
        if not capped_layers:
            for layer in active_layers:
                allocated_removal[layer] = proposed[layer]
            remaining_removal = 0.0
            break
        for layer in capped_layers:
            allocated_removal[layer] = removal_limits[layer]
            remaining_removal -= removal_limits[layer]
            active_layers.remove(layer)
    if remaining_removal > 1e-3:
        raise ValueError('Minimum retained width makes the budget infeasible')

    rows = []
    for layer in ELIGIBLE_LAYERS:
        block = mlp_blocks_by_layer[layer].module
        hidden_size = block.up_proj.in_features
        original_width = block.up_proj.out_features
        parameter_step = 3 * hidden_size
        retained_parameters = (
            original_mlp_parameters[layer]
            - allocated_removal[layer]
        )
        continuous_width = retained_parameters / parameter_step
        minimum_width = (
            1
            if minimum_retention is None
            else math.ceil(original_width * minimum_retention)
        )
        rows.append({
            'policy': policy,
            'score_name': score_name or 'uniform',
            'temperature': temperature,
            'bounded': minimum_retention is not None,
            'minimum_retention': minimum_retention,
            'layer': layer,
            'raw_importance': (
                None if scores is None else scores[layer]
            ),
            'normalized_importance': (
                None if normalized is None else normalized[layer]
            ),
            'continuous_width': continuous_width,
            'replacement_width': max(
                minimum_width, math.floor(continuous_width)
            ),
            'minimum_width': minimum_width,
            'original_width': original_width,
            'parameter_step': parameter_step,
            'original_parameters': original_mlp_parameters[layer],
        })

    parameter_steps = {row['parameter_step'] for row in rows}
    if len(parameter_steps) != 1:
        raise ValueError('Allocation requires equal per-neuron parameter cost')
    parameter_step = parameter_steps.pop()
    target_retained = eligible_mlp_parameters - target_removed
    if target_retained % parameter_step:
        raise ValueError('Target budget is not representable by whole neurons')
    target_width_total = target_retained // parameter_step
    current_width_total = sum(
        row['replacement_width'] for row in rows
    )
    while current_width_total < target_width_total:
        candidates = [
            row for row in rows
            if row['replacement_width'] < row['original_width']
        ]
        row = max(
            candidates,
            key=lambda item: (
                item['continuous_width']
                - item['replacement_width']
            ),
        )
        row['replacement_width'] += 1
        current_width_total += 1
    while current_width_total > target_width_total:
        candidates = [
            row for row in rows
            if row['replacement_width'] > row['minimum_width']
        ]
        if not candidates:
            raise ValueError('Rounded minimum widths exceed the budget')
        row = max(
            candidates,
            key=lambda item: (
                item['replacement_width']
                - item['continuous_width']
            ),
        )
        row['replacement_width'] -= 1
        current_width_total -= 1

    for row in rows:
        row['replacement_width_ratio'] = (
            row['replacement_width'] / row['original_width']
        )
        row['replacement_parameters'] = (
            row['replacement_width'] * row['parameter_step']
        )
        row['realized_sparsity'] = (
            1
            - row['replacement_parameters']
            / row['original_parameters']
        )
        row.pop('continuous_width')
        row.pop('minimum_width')
        row.pop('parameter_step')
    return pd.DataFrame(rows)


if RUN_SWIGLU_2:
    allocation_specs = [{
        'policy': 'uniform',
        'score_name': None,
        'temperature': None,
        'minimum_retention': None,
    }]
    allocation_specs.extend(
        {
            'policy': f'{score_name}_t{temperature:g}',
            'score_name': score_name,
            'temperature': temperature,
            'minimum_retention': None,
        }
        for score_name in ALLOCATION_SCORE_NAMES
        for temperature in ALLOCATION_TEMPERATURES
    )
    allocation_df = pd.concat(
        [allocate_swiglu_widths(**spec) for spec in allocation_specs],
        ignore_index=True,
    )
    allocation_summary_df = (
        allocation_df.groupby(
            ['policy', 'score_name', 'temperature', 'bounded'],
            dropna=False,
            sort=False,
            as_index=False,
        )
        .agg(
            original_mlp_parameters=('original_parameters', 'sum'),
            replacement_mlp_parameters=(
                'replacement_parameters', 'sum'
            ),
            minimum_width_ratio=('replacement_width_ratio', 'min'),
            maximum_width_ratio=('replacement_width_ratio', 'max'),
        )
    )
    allocation_summary_df['mlp_parameter_reduction_pct'] = (
        100
        * (
            1
            - allocation_summary_df['replacement_mlp_parameters']
            / allocation_summary_df['original_mlp_parameters']
        )
    )
    allocation_summary_df['model_parameters'] = (
        fixed_model_parameters
        + allocation_summary_df['replacement_mlp_parameters']
    )
    allocation_summary_df['model_parameter_reduction_pct'] = (
        100
        * (
            1
            - allocation_summary_df['model_parameters']
            / dense_footprint.parameters
        )
    )
else:
    allocation_specs = swiglu_2_artifact['configuration'][
        'allocation_specs'
    ]
    allocation_df = pd.DataFrame(
        swiglu_2_artifact['results']['allocation']
    )
    allocation_summary_df = pd.DataFrame(
        swiglu_2_artifact['results']['allocation_summary']
    )

reporting

In [ ]:
display(allocation_summary_df)

allocation_plot_df = allocation_df.copy()
allocation_plot_df['temperature_label'] = (
    allocation_plot_df['temperature']
    .map(lambda value: f'T={value:g}' if pd.notna(value) else 'uniform')
)
allocation_plot = sns.relplot(
    data=allocation_plot_df,
    x='layer',
    y='replacement_width_ratio',
    hue='temperature_label',
    style='bounded',
    col='score_name',
    col_wrap=2,
    kind='line',
    marker='o',
    height=3.7,
    aspect=1.35,
)
for axis in allocation_plot.axes.flat:
    axis.axhline(
        REFERENCE_WIDTH_RATIO,
        color='grey',
        linestyle='--',
        linewidth=1,
    )
allocation_plot.set_axis_labels(
    'Transformer block', 'Retained SwiGLU width'
)
plt.show()

#### reduced-budget allocation screening

In [ ]:
if RUN_SWIGLU_2:
    screening_fitted_operators = {
        spec['policy']: {} for spec in allocation_specs
    }
    screening_fitting_rows = []
    screening_history_rows = []
    screening_phase_started = perf_counter()

    for group_index, layer_group in enumerate(layer_groups, start=1):
        print(
            f'Screening capture {group_index}/{len(layer_groups)} '
            f'| {layer_group}'
        )
        group_paths = [
            mlp_blocks_by_layer[layer].path
            for layer in layer_group
        ]
        capture_started = perf_counter()
        training_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            screening_calibration_loader,
            SCREENING_CALIBRATION_BATCHES,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        validation_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            loaders.operator_validation,
            data_config.num_operator_validation_batches,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        runtime_rows.append({
            'stage': 'screening_capture',
            'group': group_index,
            'seconds': perf_counter() - capture_started,
        })
        group_allocation_df = allocation_df[
            allocation_df['layer'].isin(layer_group)
        ]

        for policy, policy_rows in group_allocation_df.groupby(
            'policy', sort=False
        ):
            print(f'Reduced-budget fitting | {policy}')
            for allocation in policy_rows.itertuples(index=False):
                layer = int(allocation.layer)
                block = mlp_blocks_by_layer[layer]
                training_pairs = training_pairs_by_path[block.path]
                validation_pairs = validation_pairs_by_path[block.path]
                replacement_width = int(allocation.replacement_width)
                selected_indices = teacher_neuron_rankings[layer][
                    :replacement_width
                ].sort().values
                module = GatedMLPReplacement(
                    training_pairs.hidden_size, replacement_width
                ).to(device)
                initialize_gated_mlp_from_teacher(
                    module, block.module, selected_indices
                )
                initial_metrics = evaluate_operator(
                    module,
                    validation_pairs,
                    device,
                    training_config.batch_size,
                )
                synchronize_cuda(device)
                fit_started = perf_counter()
                fit = fit_operator(
                    module,
                    training_pairs,
                    validation_pairs,
                    training_config,
                    device,
                )
                synchronize_cuda(device)
                fit_seconds = perf_counter() - fit_started
                metrics = evaluate_operator(
                    fit.module,
                    validation_pairs,
                    device,
                    training_config.batch_size,
                )
                screening_fitted_operators[policy][layer] = (
                    fit.module.to(device='cpu', dtype=model_dtype)
                )
                screening_fitting_rows.append({
                    'policy': policy,
                    'score_name': allocation.score_name,
                    'temperature': allocation.temperature,
                    'bounded': allocation.bounded,
                    'layer': layer,
                    'replacement_width': replacement_width,
                    'replacement_width_ratio': (
                        allocation.replacement_width_ratio
                    ),
                    'replacement_parameters': (
                        allocation.replacement_parameters
                    ),
                    'initial_local_relative_mse': (
                        initial_metrics.relative_mse
                    ),
                    'local_relative_mse': metrics.relative_mse,
                    'local_cosine': metrics.cosine_similarity,
                    'best_epoch': fit.best_epoch,
                    'epochs_completed': len(fit.history),
                    'fit_seconds': fit_seconds,
                })
                for epoch in fit.history:
                    screening_history_rows.append({
                        'policy': policy,
                        'layer': layer,
                        'epoch': epoch.epoch,
                        'train_mse': epoch.train_mse,
                        'validation_mse': epoch.validation_mse,
                        'learning_rate': epoch.learning_rate,
                    })

        module = fit = metrics = initial_metrics = None
        training_pairs = validation_pairs = None
        training_pairs_by_path.clear()
        validation_pairs_by_path.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        report_memory(f'After screening group {group_index}')

    screening_fitting_df = pd.DataFrame(screening_fitting_rows)
    screening_history_df = pd.DataFrame(screening_history_rows)
    runtime_rows.append({
        'stage': 'screening_fitting_total',
        'group': None,
        'seconds': perf_counter() - screening_phase_started,
    })
else:
    screening_fitted_operators = None
    screening_fitting_df = pd.DataFrame(
        swiglu_2_artifact['results']['screening_operator_fitting']
    )
    screening_history_df = pd.DataFrame(
        swiglu_2_artifact['results'][
            'screening_operator_training_history'
        ]
    )

In [ ]:
if RUN_SWIGLU_2:
    screening_model_rows = []
    screening_evaluation_started = perf_counter()
    for spec in allocation_specs:
        policy = spec['policy']
        print(f'Reduced-budget model evaluation | {policy}')
        replacements = screening_fitted_operators[policy]
        synchronize_cuda(device)
        with temporary_replacements(model, replacements):
            footprint = parameter_footprint(model)
            synchronize_cuda(device)
            evaluation_started = perf_counter()
            metrics = evaluate_language_model(
                model,
                allocation_selection_loader,
                device,
                ALLOCATION_SELECTION_BATCHES,
            )
            teacher_kl = mean_cache_loss(
                model,
                allocation_selection_teacher_cache,
                recovery_config.temperature,
                device,
            )
            synchronize_cuda(device)
            evaluation_seconds = perf_counter() - evaluation_started
        synchronize_cuda(device)
        screening_model_rows.append({
            'policy': policy,
            'score_name': spec['score_name'] or 'uniform',
            'temperature': spec['temperature'],
            'bounded': spec['minimum_retention'] is not None,
            'parameters': footprint.parameters,
            'model_parameter_reduction_pct': (
                100
                * (1 - footprint.parameters / dense_footprint.parameters)
            ),
            'teacher_kl': teacher_kl,
            'loss': metrics.loss,
            'loss_delta': (
                metrics.loss - allocation_selection_dense_metrics.loss
            ),
            'perplexity': metrics.perplexity,
            'perplexity_delta': (
                metrics.perplexity
                - allocation_selection_dense_metrics.perplexity
            ),
            'evaluation_seconds': evaluation_seconds,
        })
        for replacement in replacements.values():
            replacement.to(device='cpu', dtype=model_dtype)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    screening_model_df = pd.DataFrame(screening_model_rows)
    bi_policies = screening_model_df[
        screening_model_df['score_name'].isin(
            ['canonical_bi', 'residual_aware_mlp_bi']
        )
        & ~screening_model_df['bounded']
    ]
    kl_policies = screening_model_df[
        screening_model_df['score_name'].str.startswith(
            'singleton_kl_'
        )
    ]
    loss_policies = screening_model_df[
        screening_model_df['score_name'].str.startswith(
            'singleton_loss_delta_'
        )
    ]
    nonuniform_policies = screening_model_df[
        screening_model_df['policy'] != 'uniform'
    ]
    best_unbounded_policy = str(
        nonuniform_policies.nsmallest(1, 'teacher_kl')
        .iloc[0]['policy']
    )
    finalist_policies = list(dict.fromkeys([
        'uniform',
        str(bi_policies.nsmallest(1, 'teacher_kl').iloc[0]['policy']),
        str(
            kl_policies.nsmallest(1, 'teacher_kl')
            .iloc[0]['policy']
        ),
        str(
            loss_policies.nsmallest(1, 'loss_delta')
            .iloc[0]['policy']
        ),
        best_unbounded_policy,
    ]))
    runtime_rows.append({
        'stage': 'screening_model_evaluation_total',
        'group': None,
        'seconds': perf_counter() - screening_evaluation_started,
    })
    screening_fitted_operators.clear()
    del replacements, replacement
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    report_memory('After reduced-budget screening release')
else:
    screening_model_df = pd.DataFrame(
        swiglu_2_artifact['results']['screening_model_evaluation']
    )
    best_unbounded_policy = swiglu_2_artifact[
        'configuration'
    ]['best_unbounded_policy']
    finalist_policies = list(
        swiglu_2_artifact['configuration']['finalist_policies']
    )

reporting

In [ ]:
screening_local_summary_df = (
    screening_fitting_df.groupby('policy', as_index=False)
    .agg(
        mean_nmse=('local_relative_mse', 'mean'),
        worst_nmse=('local_relative_mse', 'max'),
        total_fit_minutes=('fit_seconds', lambda values: values.sum() / 60),
    )
)
screening_summary_df = (
    screening_model_df.merge(
        screening_local_summary_df,
        on='policy',
        validate='one_to_one',
    )
    .sort_values(['teacher_kl', 'loss_delta'])
    .reset_index(drop=True)
)
display(screening_summary_df)
print(f'Promoted policies: {finalist_policies}')

screening_plot_df = screening_summary_df.copy()
screening_plot_df['temperature_label'] = (
    screening_plot_df['temperature']
    .map(lambda value: f'T={value:g}' if pd.notna(value) else 'uniform')
)
ax = sns.scatterplot(
    data=screening_plot_df,
    x='worst_nmse',
    y='teacher_kl',
    hue='score_name',
    style='temperature_label',
    size='bounded',
    sizes=(70, 130),
)
ax.set(
    title='Reduced-budget allocation screening',
    xlabel='Worst block validation NMSE',
    ylabel='Integrated teacher KL',
)
plt.tight_layout()
plt.show()

#### minimum-width ablation

In [ ]:
if RUN_SWIGLU_2:
    best_unbounded_spec = next(
        spec for spec in allocation_specs
        if spec['policy'] == best_unbounded_policy
    )
    base_boundary_allocation = allocation_df[
        allocation_df['policy'] == best_unbounded_policy
    ].copy()
    boundary_specs = []
    boundary_frames = []

    for minimum_retention in BOUNDARY_MINIMUM_RETENTIONS:
        boundary_spec = {
            **best_unbounded_spec,
            'policy': (
                f'{best_unbounded_policy}_floor'
                f'{int(minimum_retention * 100)}'
            ),
            'minimum_retention': minimum_retention,
        }
        boundary_frame = allocate_swiglu_widths(**boundary_spec)
        boundary_widths = (
            boundary_frame['replacement_width'].tolist()
        )
        existing_widths = [
            base_boundary_allocation[
                'replacement_width'
            ].tolist(),
            *[
                frame['replacement_width'].tolist()
                for frame in boundary_frames
            ],
        ]
        if boundary_widths in existing_widths:
            continue
        boundary_specs.append(boundary_spec)
        boundary_frames.append(boundary_frame)

    new_boundary_allocation_df = (
        pd.concat(boundary_frames, ignore_index=True)
        if boundary_frames
        else allocation_df.iloc[0:0].copy()
    )
    boundary_allocation_df = pd.concat(
        [base_boundary_allocation, new_boundary_allocation_df],
        ignore_index=True,
    )
    final_allocation_df = pd.concat(
        [allocation_df, new_boundary_allocation_df],
        ignore_index=True,
    )
else:
    boundary_specs = swiglu_2_artifact['configuration'][
        'boundary_specs'
    ]
    boundary_allocation_df = pd.DataFrame(
        swiglu_2_artifact['results']['boundary_allocation']
    )
    new_boundary_allocation_df = boundary_allocation_df[
        boundary_allocation_df['policy'] != best_unbounded_policy
    ].reset_index(drop=True)
    final_allocation_df = pd.concat(
        [allocation_df, new_boundary_allocation_df],
        ignore_index=True,
    )

In [ ]:
if RUN_SWIGLU_2:
    boundary_fitted_operators = {
        spec['policy']: {} for spec in boundary_specs
    }
    boundary_fitting_rows = []
    boundary_history_rows = []
    boundary_model_rows = []
    boundary_phase_started = perf_counter()

    for group_index, layer_group in enumerate(layer_groups, start=1):
        if not boundary_specs:
            break
        print(
            f'Boundary capture {group_index}/{len(layer_groups)} '
            f'| {layer_group}'
        )
        group_paths = [
            mlp_blocks_by_layer[layer].path
            for layer in layer_group
        ]
        training_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            screening_calibration_loader,
            SCREENING_CALIBRATION_BATCHES,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        validation_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            loaders.operator_validation,
            data_config.num_operator_validation_batches,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        group_allocation_df = new_boundary_allocation_df[
            new_boundary_allocation_df['layer'].isin(layer_group)
        ]

        for policy, policy_rows in group_allocation_df.groupby(
            'policy', sort=False
        ):
            print(f'Boundary screening | {policy}')
            for allocation in policy_rows.itertuples(index=False):
                layer = int(allocation.layer)
                block = mlp_blocks_by_layer[layer]
                training_pairs = training_pairs_by_path[block.path]
                validation_pairs = validation_pairs_by_path[block.path]
                replacement_width = int(allocation.replacement_width)
                selected_indices = teacher_neuron_rankings[layer][
                    :replacement_width
                ].sort().values
                module = GatedMLPReplacement(
                    training_pairs.hidden_size, replacement_width
                ).to(device)
                initialize_gated_mlp_from_teacher(
                    module, block.module, selected_indices
                )
                initial_metrics = evaluate_operator(
                    module,
                    validation_pairs,
                    device,
                    training_config.batch_size,
                )
                synchronize_cuda(device)
                fit_started = perf_counter()
                fit = fit_operator(
                    module,
                    training_pairs,
                    validation_pairs,
                    training_config,
                    device,
                )
                synchronize_cuda(device)
                fit_seconds = perf_counter() - fit_started
                metrics = evaluate_operator(
                    fit.module,
                    validation_pairs,
                    device,
                    training_config.batch_size,
                )
                boundary_fitted_operators[policy][layer] = (
                    fit.module.to(device='cpu', dtype=model_dtype)
                )
                boundary_fitting_rows.append({
                    'policy': policy,
                    'layer': layer,
                    'minimum_retention': (
                        allocation.minimum_retention
                    ),
                    'replacement_width': replacement_width,
                    'replacement_width_ratio': (
                        allocation.replacement_width_ratio
                    ),
                    'replacement_parameters': (
                        allocation.replacement_parameters
                    ),
                    'initial_local_relative_mse': (
                        initial_metrics.relative_mse
                    ),
                    'local_relative_mse': metrics.relative_mse,
                    'local_cosine': metrics.cosine_similarity,
                    'best_epoch': fit.best_epoch,
                    'epochs_completed': len(fit.history),
                    'fit_seconds': fit_seconds,
                })
                for epoch in fit.history:
                    boundary_history_rows.append({
                        'policy': policy,
                        'layer': layer,
                        'epoch': epoch.epoch,
                        'train_mse': epoch.train_mse,
                        'validation_mse': epoch.validation_mse,
                        'learning_rate': epoch.learning_rate,
                    })

        module = fit = metrics = initial_metrics = None
        training_pairs = validation_pairs = None
        training_pairs_by_path.clear()
        validation_pairs_by_path.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    for spec in boundary_specs:
        policy = spec['policy']
        replacements = boundary_fitted_operators[policy]
        synchronize_cuda(device)
        evaluation_started = perf_counter()
        with temporary_replacements(model, replacements):
            footprint = parameter_footprint(model)
            metrics = evaluate_language_model(
                model,
                allocation_selection_loader,
                device,
                ALLOCATION_SELECTION_BATCHES,
            )
            teacher_kl = mean_cache_loss(
                model,
                allocation_selection_teacher_cache,
                recovery_config.temperature,
                device,
            )
        synchronize_cuda(device)
        boundary_model_rows.append({
            'policy': policy,
            'score_name': spec['score_name'],
            'temperature': spec['temperature'],
            'bounded': True,
            'minimum_retention': spec['minimum_retention'],
            'parameters': footprint.parameters,
            'model_parameter_reduction_pct': (
                100
                * (1 - footprint.parameters / dense_footprint.parameters)
            ),
            'teacher_kl': teacher_kl,
            'loss': metrics.loss,
            'loss_delta': (
                metrics.loss - allocation_selection_dense_metrics.loss
            ),
            'perplexity': metrics.perplexity,
            'perplexity_delta': (
                metrics.perplexity
                - allocation_selection_dense_metrics.perplexity
            ),
            'evaluation_seconds': perf_counter() - evaluation_started,
        })
        for replacement in replacements.values():
            replacement.to(device='cpu', dtype=model_dtype)

    boundary_fitting_df = pd.DataFrame(boundary_fitting_rows)
    boundary_history_df = pd.DataFrame(boundary_history_rows)
    boundary_model_df = pd.DataFrame(boundary_model_rows)
    base_boundary_model_df = screening_model_df[
        screening_model_df['policy'] == best_unbounded_policy
    ].copy()
    base_boundary_model_df['minimum_retention'] = None
    boundary_local_df = (
        pd.concat([
            screening_fitting_df[
                screening_fitting_df['policy']
                == best_unbounded_policy
            ],
            boundary_fitting_df,
        ], ignore_index=True)
        .groupby('policy', as_index=False)
        .agg(
            mean_nmse=('local_relative_mse', 'mean'),
            worst_nmse=('local_relative_mse', 'max'),
        )
    )
    boundary_width_summary_df = (
        boundary_allocation_df.groupby(
            'policy', as_index=False
        )
        .agg(
            minimum_width_ratio=(
                'replacement_width_ratio', 'min'
            ),
            maximum_width_ratio=(
                'replacement_width_ratio', 'max'
            ),
        )
    )
    boundary_screening_df = (
        pd.concat(
            [base_boundary_model_df, boundary_model_df],
            ignore_index=True,
        )
        .merge(boundary_local_df, on='policy', validate='one_to_one')
        .merge(
            boundary_width_summary_df,
            on='policy',
            validate='one_to_one',
        )
    )
    best_bounded_policy = (
        None
        if boundary_model_df.empty
        else str(
            boundary_model_df.nsmallest(1, 'teacher_kl')
            .iloc[0]['policy']
        )
    )
    if best_bounded_policy is not None:
        finalist_policies = list(dict.fromkeys([
            *finalist_policies,
            best_bounded_policy,
        ]))
    runtime_rows.append({
        'stage': 'minimum_width_ablation_total',
        'group': None,
        'seconds': perf_counter() - boundary_phase_started,
    })
    boundary_fitted_operators.clear()
    if boundary_specs:
        del replacements, replacement
    del allocation_selection_teacher_cache
    allocation_selection_teacher_cache = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    report_memory('After minimum-width ablation release')
else:
    boundary_fitting_df = pd.DataFrame(
        swiglu_2_artifact['results']['boundary_operator_fitting']
    )
    boundary_history_df = pd.DataFrame(
        swiglu_2_artifact['results'][
            'boundary_operator_training_history'
        ]
    )
    boundary_model_df = pd.DataFrame(
        swiglu_2_artifact['results']['boundary_model_evaluation']
    )
    boundary_screening_df = pd.DataFrame(
        swiglu_2_artifact['results']['boundary_screening_comparison']
    )
    best_bounded_policy = swiglu_2_artifact[
        'configuration'
    ]['best_bounded_policy']

reporting

In [ ]:
display(boundary_screening_df)

if len(boundary_screening_df) > 1:
    boundary_plot_df = boundary_allocation_df.copy()
    boundary_plot_df['minimum_retention_label'] = (
        boundary_plot_df['minimum_retention']
        .map(
            lambda value: (
                f'{value:.0%}' if pd.notna(value) else 'none'
            )
        )
    )
    ax = sns.lineplot(
        data=boundary_plot_df,
        x='layer',
        y='replacement_width_ratio',
        hue='minimum_retention_label',
        marker='o',
    )
    ax.set(
        title=f'Minimum-width ablation | {best_unbounded_policy}',
        xlabel='Transformer block',
        ylabel='Retained SwiGLU width',
    )
    plt.tight_layout()
    plt.show()
else:
    print('Candidate floors did not change the winning allocation.')

#### full-budget finalists

In [ ]:
if RUN_SWIGLU_2:
    full_fitted_operators = {'uniform': reference_operators}
    full_fitting_rows = []
    for row in reference_fitting_df.to_dict(orient='records'):
        full_fitting_rows.append({
            **row,
            'score_name': 'uniform',
            'temperature': None,
            'bounded': False,
        })
    full_history_rows = reference_history_df.to_dict(
        orient='records'
    )
    nonuniform_finalists = [
        policy for policy in finalist_policies
        if policy != 'uniform'
    ]
    for policy in nonuniform_finalists:
        full_fitted_operators[policy] = {}

    full_fit_phase_started = perf_counter()
    for group_index, layer_group in enumerate(layer_groups, start=1):
        print(
            f'Finalist capture {group_index}/{len(layer_groups)} '
            f'| {layer_group}'
        )
        group_paths = [
            mlp_blocks_by_layer[layer].path
            for layer in layer_group
        ]
        capture_started = perf_counter()
        training_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            loaders.calibration,
            data_config.num_calibration_batches,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        validation_pairs_by_path = collect_modules_io(
            model,
            group_paths,
            loaders.operator_validation,
            data_config.num_operator_validation_batches,
            device,
            storage_device=capture_config.storage_device,
            storage_dtype=model_dtype,
        )
        runtime_rows.append({
            'stage': 'finalist_capture',
            'group': group_index,
            'seconds': perf_counter() - capture_started,
        })
        group_allocation_df = final_allocation_df[
            final_allocation_df['layer'].isin(layer_group)
            & final_allocation_df['policy'].isin(
                nonuniform_finalists
            )
        ]

        for policy, policy_rows in group_allocation_df.groupby(
            'policy', sort=False
        ):
            print(f'Full-budget fitting | {policy}')
            for allocation in policy_rows.itertuples(index=False):
                layer = int(allocation.layer)
                block = mlp_blocks_by_layer[layer]
                training_pairs = training_pairs_by_path[block.path]
                validation_pairs = validation_pairs_by_path[block.path]
                replacement_width = int(allocation.replacement_width)
                selected_indices = teacher_neuron_rankings[layer][
                    :replacement_width
                ].sort().values
                module = GatedMLPReplacement(
                    training_pairs.hidden_size, replacement_width
                ).to(device)
                initialize_gated_mlp_from_teacher(
                    module, block.module, selected_indices
                )
                initial_metrics = evaluate_operator(
                    module,
                    validation_pairs,
                    device,
                    training_config.batch_size,
                )
                synchronize_cuda(device)
                fit_started = perf_counter()
                fit = fit_operator(
                    module,
                    training_pairs,
                    validation_pairs,
                    training_config,
                    device,
                )
                synchronize_cuda(device)
                fit_seconds = perf_counter() - fit_started
                metrics = evaluate_operator(
                    fit.module,
                    validation_pairs,
                    device,
                    training_config.batch_size,
                )
                full_fitted_operators[policy][layer] = fit.module.to(
                    device='cpu', dtype=model_dtype
                )
                full_fitting_rows.append({
                    'policy': policy,
                    'score_name': allocation.score_name,
                    'temperature': allocation.temperature,
                    'bounded': allocation.bounded,
                    'layer': layer,
                    'replacement_width': replacement_width,
                    'replacement_width_ratio': (
                        allocation.replacement_width_ratio
                    ),
                    'replacement_parameters': (
                        allocation.replacement_parameters
                    ),
                    'initial_local_relative_mse': (
                        initial_metrics.relative_mse
                    ),
                    'local_relative_mse': metrics.relative_mse,
                    'local_cosine': metrics.cosine_similarity,
                    'best_epoch': fit.best_epoch,
                    'epochs_completed': len(fit.history),
                    'fit_seconds': fit_seconds,
                })
                for epoch in fit.history:
                    full_history_rows.append({
                        'policy': policy,
                        'layer': layer,
                        'epoch': epoch.epoch,
                        'train_mse': epoch.train_mse,
                        'validation_mse': epoch.validation_mse,
                        'learning_rate': epoch.learning_rate,
                    })

        module = fit = metrics = initial_metrics = None
        training_pairs = validation_pairs = None
        training_pairs_by_path.clear()
        validation_pairs_by_path.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        report_memory(f'After finalist group {group_index}')

    full_fitting_df = pd.DataFrame(full_fitting_rows)
    full_history_df = pd.DataFrame(full_history_rows)
    runtime_rows.append({
        'stage': 'finalist_fitting_total',
        'group': None,
        'seconds': perf_counter() - full_fit_phase_started,
    })
else:
    full_fitted_operators = None
    full_fitting_df = pd.DataFrame(
        swiglu_2_artifact['results']['finalist_operator_fitting']
    )
    full_history_df = pd.DataFrame(
        swiglu_2_artifact['results'][
            'finalist_operator_training_history'
        ]
    )

In [ ]:
if RUN_SWIGLU_2:
    final_validation_teacher_cache = cache_teacher_logits(
        model,
        loaders.model_validation,
        data_config.num_model_validation_batches,
        device,
        recovery_config.cache_dtype,
    )
    dense_reference_loss = float(dense_reference_df.iloc[0]['loss'])
    dense_reference_perplexity = float(
        dense_reference_df.iloc[0]['perplexity']
    )
    pre_recovery_rows = []
    pre_evaluation_started = perf_counter()
    for policy in finalist_policies:
        print(f'Full validation before recovery | {policy}')
        replacements = full_fitted_operators[policy]
        synchronize_cuda(device)
        evaluation_started = perf_counter()
        with temporary_replacements(model, replacements):
            footprint = parameter_footprint(model)
            metrics = evaluate_language_model(
                model,
                loaders.model_validation,
                device,
                data_config.num_model_validation_batches,
            )
            teacher_kl = mean_cache_loss(
                model,
                final_validation_teacher_cache,
                recovery_config.temperature,
                device,
            )
        synchronize_cuda(device)
        policy_allocation = final_allocation_df[
            final_allocation_df['policy'] == policy
        ]
        pre_recovery_rows.append({
            'policy': policy,
            'phase': 'pre_recovery',
            'parameters': footprint.parameters,
            'model_parameter_reduction_pct': (
                100
                * (1 - footprint.parameters / dense_footprint.parameters)
            ),
            'mlp_parameter_reduction_pct': (
                100
                * (
                    1
                    - policy_allocation['replacement_parameters'].sum()
                    / policy_allocation['original_parameters'].sum()
                )
            ),
            'teacher_kl': teacher_kl,
            'loss': metrics.loss,
            'loss_delta': metrics.loss - dense_reference_loss,
            'perplexity': metrics.perplexity,
            'perplexity_delta': (
                metrics.perplexity - dense_reference_perplexity
            ),
            'evaluation_seconds': perf_counter() - evaluation_started,
        })
        for replacement in replacements.values():
            replacement.to(device='cpu', dtype=model_dtype)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pre_recovery_df = pd.DataFrame(pre_recovery_rows)
    runtime_rows.append({
        'stage': 'finalist_pre_recovery_evaluation_total',
        'group': None,
        'seconds': perf_counter() - pre_evaluation_started,
    })
else:
    final_validation_teacher_cache = None
    pre_recovery_df = pd.DataFrame(
        swiglu_2_artifact['results']['model_evaluation']
    ).query("phase == 'pre_recovery'").reset_index(drop=True)

reporting

In [ ]:
full_local_summary_df = (
    full_fitting_df.groupby('policy', as_index=False)
    .agg(
        mean_nmse=('local_relative_mse', 'mean'),
        worst_nmse=('local_relative_mse', 'max'),
        total_fit_minutes=('fit_seconds', lambda values: values.sum() / 60),
    )
)
pre_recovery_summary_df = (
    pre_recovery_df.merge(
        full_local_summary_df,
        on='policy',
        validate='one_to_one',
    )
    .sort_values('teacher_kl')
    .reset_index(drop=True)
)
display(pre_recovery_summary_df)

#### model-wide recovery

In [ ]:
if RUN_SWIGLU_2:
    report_memory('Before recovery caches')
    recovery_teacher_cache = cache_teacher_logits(
        model,
        loaders.recovery,
        data_config.num_recovery_batches,
        device,
        recovery_config.cache_dtype,
    )
    recovery_validation_teacher_cache = cache_teacher_logits(
        model,
        loaders.recovery_validation,
        data_config.num_recovery_validation_batches,
        device,
        recovery_config.cache_dtype,
    )
    report_memory('After recovery caches')
    post_recovery_rows = []
    recovery_rows = []
    recovery_phase_started = perf_counter()

    for policy in finalist_policies:
        print(f'Model-wide recovery | {policy}')
        replacements = {
            layer: deepcopy(full_fitted_operators[policy][layer])
            for layer in ELIGIBLE_LAYERS
        }
        with temporary_replacements(model, replacements) as manifest:
            target_paths = [record.path for record in manifest.records]
            synchronize_cuda(device)
            recovery_started = perf_counter()
            recovery = recover_replacements(
                model,
                recovery_teacher_cache,
                recovery_validation_teacher_cache,
                target_paths,
                recovery_config,
                device,
            )
            synchronize_cuda(device)
            recovery_seconds = perf_counter() - recovery_started
            footprint = parameter_footprint(model)
            synchronize_cuda(device)
            evaluation_started = perf_counter()
            metrics = evaluate_language_model(
                model,
                loaders.model_validation,
                device,
                data_config.num_model_validation_batches,
            )
            teacher_kl = mean_cache_loss(
                model,
                final_validation_teacher_cache,
                recovery_config.temperature,
                device,
            )
            synchronize_cuda(device)
            evaluation_seconds = perf_counter() - evaluation_started
            policy_allocation = final_allocation_df[
                final_allocation_df['policy'] == policy
            ]
            post_recovery_rows.append({
                'policy': policy,
                'phase': 'post_recovery',
                'parameters': footprint.parameters,
                'model_parameter_reduction_pct': (
                    100
                    * (
                        1
                        - footprint.parameters
                        / dense_footprint.parameters
                    )
                ),
                'mlp_parameter_reduction_pct': (
                    100
                    * (
                        1
                        - policy_allocation[
                            'replacement_parameters'
                        ].sum()
                        / policy_allocation[
                            'original_parameters'
                        ].sum()
                    )
                ),
                'teacher_kl': teacher_kl,
                'loss': metrics.loss,
                'loss_delta': metrics.loss - dense_reference_loss,
                'perplexity': metrics.perplexity,
                'perplexity_delta': (
                    metrics.perplexity - dense_reference_perplexity
                ),
                'evaluation_seconds': evaluation_seconds,
                'recovery_seconds': recovery_seconds,
            })
            for epoch in recovery.history:
                recovery_rows.append({
                    'policy': policy,
                    'epoch': epoch.epoch,
                    'train_kl': epoch.train_kl,
                    'validation_kl': epoch.validation_kl,
                    'best_epoch': recovery.best_epoch,
                    'recovery_seconds': recovery_seconds,
                })

        for replacement in replacements.values():
            replacement.to(device='cpu', dtype=model_dtype)
        del replacements, replacement
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        report_memory(f'After recovery | {policy}')

    model_evaluation_df = pd.concat(
        [pre_recovery_df, pd.DataFrame(post_recovery_rows)],
        ignore_index=True,
    )
    recovery_history_df = pd.DataFrame(recovery_rows)
    runtime_rows.append({
        'stage': 'model_wide_recovery_total',
        'group': None,
        'seconds': perf_counter() - recovery_phase_started,
    })
    del recovery_teacher_cache
    del recovery_validation_teacher_cache
    del final_validation_teacher_cache
    final_validation_teacher_cache = None
    full_fitted_operators.clear()
    reference_operators.clear()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    report_memory('After recovery cache release')
else:
    model_evaluation_df = pd.DataFrame(
        swiglu_2_artifact['results']['model_evaluation']
    )
    recovery_history_df = pd.DataFrame(
        swiglu_2_artifact['results']['recovery_history']
    )

reporting

In [ ]:
pre_summary = (
    model_evaluation_df.query("phase == 'pre_recovery'")
    [[
        'policy',
        'parameters',
        'model_parameter_reduction_pct',
        'mlp_parameter_reduction_pct',
        'teacher_kl',
        'loss',
        'loss_delta',
        'perplexity',
        'perplexity_delta',
        'evaluation_seconds',
    ]]
    .rename(columns={
        'teacher_kl': 'pre_teacher_kl',
        'loss': 'pre_loss',
        'loss_delta': 'pre_loss_delta',
        'perplexity': 'pre_perplexity',
        'perplexity_delta': 'pre_perplexity_delta',
        'evaluation_seconds': 'pre_evaluation_seconds',
    })
)
post_summary = (
    model_evaluation_df.query("phase == 'post_recovery'")
    [[
        'policy',
        'teacher_kl',
        'loss',
        'loss_delta',
        'perplexity',
        'perplexity_delta',
        'evaluation_seconds',
        'recovery_seconds',
    ]]
    .rename(columns={
        'teacher_kl': 'post_teacher_kl',
        'loss': 'post_loss',
        'loss_delta': 'post_loss_delta',
        'perplexity': 'post_perplexity',
        'perplexity_delta': 'post_perplexity_delta',
        'evaluation_seconds': 'post_evaluation_seconds',
    })
)
final_allocation_summary_df = (
    final_allocation_df[
        final_allocation_df['policy'].isin(finalist_policies)
    ]
    .groupby('policy', as_index=False)
    .agg(
        score_name=('score_name', 'first'),
        temperature=('temperature', 'first'),
        bounded=('bounded', 'first'),
        minimum_width_ratio=('replacement_width_ratio', 'min'),
        maximum_width_ratio=('replacement_width_ratio', 'max'),
    )
)
final_summary_df = (
    pre_summary.merge(post_summary, on='policy', validate='one_to_one')
    .merge(full_local_summary_df, on='policy', validate='one_to_one')
    .merge(
        final_allocation_summary_df,
        on='policy',
        validate='one_to_one',
    )
)
final_summary_df['kl_recovered_pct'] = (
    100
    * (
        1
        - final_summary_df['post_teacher_kl']
        / final_summary_df['pre_teacher_kl']
    )
)
final_summary_df['loss_gap_recovered_pct'] = (
    100
    * (
        final_summary_df['pre_loss_delta']
        - final_summary_df['post_loss_delta']
    )
    / final_summary_df['pre_loss_delta']
)
final_summary_df = final_summary_df.sort_values(
    'post_teacher_kl'
).reset_index(drop=True)
winning_policy = str(final_summary_df.iloc[0]['policy'])
print(f'Winning policy by post-recovery KL: {winning_policy}')
display(final_summary_df[[
    'policy',
    'score_name',
    'temperature',
    'bounded',
    'minimum_width_ratio',
    'maximum_width_ratio',
    'model_parameter_reduction_pct',
    'mlp_parameter_reduction_pct',
    'mean_nmse',
    'worst_nmse',
    'pre_teacher_kl',
    'post_teacher_kl',
    'kl_recovered_pct',
    'pre_perplexity',
    'post_perplexity',
    'loss_gap_recovered_pct',
    'total_fit_minutes',
    'recovery_seconds',
]])

figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for axis, metric, label in zip(
    axes,
    ('teacher_kl', 'perplexity'),
    ('Teacher KL', 'Perplexity'),
):
    sns.barplot(
        data=model_evaluation_df,
        x='policy',
        y=metric,
        hue='phase',
        ax=axis,
    )
    axis.set_xlabel('Allocation policy')
    axis.set_ylabel(label)
    axis.tick_params(axis='x', rotation=30)
figure.suptitle('Full-budget finalists before and after recovery')
figure.tight_layout()
plt.show()

#### artifact

In [ ]:
def json_records(frame):
    return json.loads(
        frame.to_json(orient='records', double_precision=15)
    )


if RUN_SWIGLU_2:
    runtime_rows = [
        row for row in runtime_rows
        if row['stage'] != 'notebook_total'
    ]
    runtime_rows.append({
        'stage': 'notebook_total',
        'group': None,
        'seconds': perf_counter() - RUN_STARTED,
    })
    runtime_df = pd.DataFrame(runtime_rows)
    swiglu_2_artifact = {
        'schema_version': SWIGLU_2_ARTIFACT_SCHEMA_VERSION,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'environment': environment_record(),
        'configuration': {
            'model': {
                **asdict(model_config),
                'resolved_revision': getattr(
                    model.config, '_commit_hash', None
                ),
            },
            'data': asdict(data_config),
            'capture': {
                **asdict(capture_config),
                'storage_dtype': str(model_dtype).removeprefix(
                    'torch.'
                ),
                'module_group_size': CAPTURE_GROUP_SIZE,
                'disk_io': False,
            },
            'operator_training': asdict(training_config),
            'recovery': {
                **asdict(recovery_config),
                'trainable_scope': 'replacement_only',
            },
            'reference_artifact': {
                'path': str(REFERENCE_ARTIFACT_PATH),
                'schema_version': (
                    reference_artifact['schema_version']
                ),
            },
            'partition_batches': PARTITION_BATCHES,
            'partition_order': list(PARTITION_BATCHES),
            'screening_calibration_batches': (
                SCREENING_CALIBRATION_BATCHES
            ),
            'probe_width_ratios': list(PROBE_WIDTH_RATIOS),
            'reference_width_ratio': REFERENCE_WIDTH_RATIO,
            'width_sweep_ratios': list(WIDTH_SWEEP_RATIOS),
            'allocation_score_names': list(ALLOCATION_SCORE_NAMES),
            'allocation_temperatures': list(
                ALLOCATION_TEMPERATURES
            ),
            'boundary_minimum_retentions': list(
                BOUNDARY_MINIMUM_RETENTIONS
            ),
            'target_mlp_sparsity': TARGET_MLP_SPARSITY,
            'eligible_layers': list(ELIGIBLE_LAYERS),
            'protected_layers': list(PROTECTED_LAYERS),
            'representative_blocks': representative_blocks,
            'allocation_specs': allocation_specs,
            'boundary_specs': boundary_specs,
            'best_unbounded_policy': best_unbounded_policy,
            'best_bounded_policy': best_bounded_policy,
            'finalist_policies': finalist_policies,
            'winning_policy': winning_policy,
        },
        'results': {
            'dense_reference': json_records(dense_reference_df),
            'probe_operator_fitting': json_records(
                probe_fitting_df
            ),
            'probe_operator_training_history': json_records(
                probe_history_df
            ),
            'probe_model_impact': json_records(probe_impact_df),
            'candidate_scores': json_records(candidate_score_df),
            'representative_block_membership': json_records(
                representative_membership_df
            ),
            'width_study': json_records(width_study_df),
            'width_study_history': json_records(
                width_study_history_df
            ),
            'allocation': json_records(allocation_df),
            'allocation_summary': json_records(
                allocation_summary_df
            ),
            'screening_operator_fitting': json_records(
                screening_fitting_df
            ),
            'screening_operator_training_history': json_records(
                screening_history_df
            ),
            'screening_model_evaluation': json_records(
                screening_model_df
            ),
            'boundary_allocation': json_records(
                boundary_allocation_df
            ),
            'boundary_operator_fitting': json_records(
                boundary_fitting_df
            ),
            'boundary_operator_training_history': json_records(
                boundary_history_df
            ),
            'boundary_model_evaluation': json_records(
                boundary_model_df
            ),
            'boundary_screening_comparison': json_records(
                boundary_screening_df
            ),
            'finalist_operator_fitting': json_records(
                full_fitting_df
            ),
            'finalist_operator_training_history': json_records(
                full_history_df
            ),
            'model_evaluation': json_records(
                model_evaluation_df
            ),
            'recovery_history': json_records(
                recovery_history_df
            ),
            'final_summary': json_records(final_summary_df),
            'runtime': json_records(runtime_df),
        },
    }
    SWIGLU_2_ARTIFACT_PATH.parent.mkdir(
        parents=True, exist_ok=True
    )
    SWIGLU_2_ARTIFACT_PATH.write_text(
        json.dumps(
            swiglu_2_artifact,
            indent=2,
            allow_nan=False,
        ),
        encoding='utf-8',
    )
    print(f'Saved artifact to {SWIGLU_2_ARTIFACT_PATH}')
else:
    runtime_df = pd.DataFrame(
        swiglu_2_artifact['results']['runtime']
    )
    print(f'Loaded artifact from {SWIGLU_2_ARTIFACT_PATH}')